# Archetype 1 — Base Household

## Purpose

Computes the annual Total Cost of Electricity (TCoE) for **Archetype 1: Base** (standard household demand, no flexible devices) across all 3 operational strategies and 7 DSOs → **21 runs**.

Because Archetype 1 has no flexible device, the grid draw equals base demand in every timestep regardless of strategy. All three strategies (no-flex, DT-flex, TCoE-flex) therefore produce **identical schedules**; only the DSO bill differs per run.

## Input files (`inputs/`)

| File | Content |
|---|---|
| `base_demand_h25_4500kwh_2026_15min.csv` | 35,040-slot BDEW H25 demand (kWh/slot) |
| `spot_prices_de_lu_2025_15min.csv` | Day-ahead spot price (ct/kWh, 15-min) |
| `dso_tariffs_residential_2026.csv` | Residential base tariffs for 7 DSOs |
| `residential_taxes_2026.csv` | German levies (ct/kWh, pre-VAT) |

## Output (`outputs/`)

`results_base_2026.csv` — 21 rows × cost-component columns (see Step 3).

## Billing convention

All bill components are **net of VAT** except where noted:
- Levies (pre-VAT) are included in the net subtotal; a single 19% VAT is applied to the full subtotal.
- `cost_smart_operating_net_eur = 0` for Base (no iMSys / smart-operating requirement).
- No §14a Module 1/2/3 rebate applies (no registered controllable device).

## Thesis reference

Chapter 5, Section 5.1 — Results: Archetype 1 (Base)

In [1]:
import pandas as pd
from pathlib import Path

# ── Path configuration ─────────────────────────────────────────────────────────
def find_repo_root(marker='README.md'):
    p = Path('__file__').resolve().parent
    for candidate in [p, *p.parents]:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(f'Repo root (containing {marker}) not found.')

REPO_ROOT  = find_repo_root()
INPUTS     = REPO_ROOT / 'inputs'
OUTPUTS    = REPO_ROOT / 'outputs'
OUTPUTS.mkdir(exist_ok=True)

print(f'Repo root : {REPO_ROOT}')
print(f'Inputs    : {INPUTS}')
print(f'Outputs   : {OUTPUTS}')

Repo root : /Users/juliusrucha/Library/CloudStorage/GoogleDrive-rucha.julius@gmail.com/Meine Ablage/Master_Thesis_2026/07_Thesis_GitHub
Inputs    : /Users/juliusrucha/Library/CloudStorage/GoogleDrive-rucha.julius@gmail.com/Meine Ablage/Master_Thesis_2026/07_Thesis_GitHub/inputs
Outputs   : /Users/juliusrucha/Library/CloudStorage/GoogleDrive-rucha.julius@gmail.com/Meine Ablage/Master_Thesis_2026/07_Thesis_GitHub/outputs


## Step 1 — Load inputs

In [2]:
# Base demand (35,040 slots × 15 min)
demand = pd.read_csv(INPUTS / 'base_demand_h25_4500kwh_2026_15min.csv',
                     parse_dates=['timestamp'])

# Spot prices (ct/kWh)
spot   = pd.read_csv(INPUTS / 'spot_prices_de_lu_2025_15min.csv',
                     parse_dates=['timestamp'])

# Merge on datetime
df = demand.merge(spot, on='timestamp', how='inner')
df = df.rename(columns={'demand_kwh': 'E_grid_kWh', 'price_ct_kWh': 'c_spot_ct_kWh'})

# DSO tariffs
dso_tariffs = pd.read_csv(INPUTS / 'dso_tariffs_residential_2026.csv')
DSO_LIST    = list(dso_tariffs['DSO'])

# Levies
taxes = pd.read_csv(INPUTS / 'residential_taxes_2026.csv')
tax_row = taxes.loc[taxes['region'] == 'DE'].iloc[0]
TAX_PRE_VAT_CT_KWH = float(tax_row['Total_no_VAT_ct_kWh'])  # pre-VAT; VAT applied via subtotal

VAT_RATE  = 0.19
ARCHETYPE = 1
STRATEGIES = ['no_flex', 'dt_flex', 'tcoe_flex']

print(f'Time steps        : {len(df):,}')
print(f'Annual demand     : {df["E_grid_kWh"].sum():.2f} kWh')
print(f'Levies (pre-VAT)  : {TAX_PRE_VAT_CT_KWH} ct/kWh')
print(f'DSOs              : {DSO_LIST}')

Time steps        : 35,040
Annual demand     : 4500.00 kWh
Levies (pre-VAT)  : 6.64 ct/kWh
DSOs              : ['Westnetz', 'Bayernwerk', 'E.DIS', 'Netze BW', 'Stromnetz Berlin', 'SH Netz', 'MITNETZ STROM']


## Step 2 — Compute billing (21 runs)

For each strategy × DSO, the bill is:

$$\text{TCoE} = \underbrace{(C_{spot} + C_{DSO,fix} + C_{DSO,vol} + C_{levies,pre-VAT})}_{\text{net subtotal}} \times 1.19$$

No §14a rebate, no PV feed-in, no device-related costs.

In [3]:
annual_kwh      = df['E_grid_kWh'].sum()
energy_cost_net = (df['E_grid_kWh'] * df['c_spot_ct_kWh']).sum() / 100.0  # same for all DSOs
levies_pre_vat  = annual_kwh * TAX_PRE_VAT_CT_KWH / 100.0

records = []
run_id  = 0
for strategy in STRATEGIES:
    for dso_id in DSO_LIST:
        run_id += 1
        dso = dso_tariffs.loc[dso_tariffs['DSO'] == dso_id].iloc[0]

        c_fix_net = float(dso['Grundpreis_EUR_year'])
        c_vol_net = annual_kwh * float(dso['Arbeitspreis_ct_kWh']) / 100.0

        subtotal_net = energy_cost_net + c_fix_net + c_vol_net + levies_pre_vat
        vat          = VAT_RATE * subtotal_net
        total_tcoe   = subtotal_net + vat

        records.append({
            'run_id'                      : run_id,
            'scenario_id'                 : f'{strategy}_{ARCHETYPE}_{dso_id}_2026',
            'strategy'                    : strategy,
            'household_archetype'         : ARCHETYPE,
            'dso_id'                      : dso_id,
            'annual_energy_kWh'           : round(annual_kwh, 2),
            'cost_energy_net_eur'         : round(energy_cost_net, 2),
            'cost_dso_fixed_net_eur'      : round(c_fix_net, 2),
            'cost_dso_volumetric_net_eur' : round(c_vol_net, 2),
            'cost_smart_operating_net_eur': 0.0,
            'cost_levies_pre_vat_eur'      : round(levies_pre_vat, 2),
            'subtotal_net_eur'             : round(subtotal_net, 2),
            'vat_eur'                      : round(vat, 2),
            'total_tcoe_eur'              : round(total_tcoe, 2),
        })

results = pd.DataFrame(records)
results

,run_id,scenario_id,strategy,household_archetype,dso_id,annual_energy_kWh,cost_energy_net_eur,cost_dso_fixed_net_eur,cost_dso_volumetric_net_eur,cost_smart_operating_net_eur,cost_levies_pre_vat_eur,subtotal_net_eur,vat_eur,total_tcoe_eur
0,1,no_flex_1_Westnetz_2026,no_flex,1,Westnetz,4500.0,419.63,80.30,428.85,0.0,298.8,1227.58,233.24,1460.82
1,2,no_flex_1_Bayernwerk_2026,no_flex,1,Bayernwerk,4500.0,419.63,98.55,212.40,0.0,298.8,1029.38,195.58,1224.96
2,3,no_flex_1_E.DIS_2026,no_flex,1,E.DIS,4500.0,419.63,76.65,246.15,0.0,298.8,1041.23,197.83,1239.06
3,4,no_flex_1_Netze BW_2026,no_flex,1,Netze BW,4500.0,419.63,84.00,340.65,0.0,298.8,1143.08,217.18,1360.26
4,5,no_flex_1_Stromnetz Berlin_2026,no_flex,1,Stromnetz Berlin,4500.0,419.63,33.36,335.70,0.0,298.8,1087.49,206.62,1294.11
5,6,no_flex_1_SH Netz_2026,no_flex,1,SH Netz,4500.0,419.63,94.90,288.00,0.0,298.8,1101.33,209.25,1310.58
6,7,no_flex_1_MITNETZ STROM_2026,no_flex,1,MITNETZ STROM,4500.0,419.63,73.00,283.95,0.0,298.8,1075.38,204.32,1279.70
7,8,dt_flex_1_Westnetz_2026,dt_flex,1,Westnetz,4500.0,419.63,80.30,428.85,0.0,298.8,1227.58,233.24,1460.82
8,9,dt_flex_1_Bayernwerk_2026,dt_flex,1,Bayernwerk,4500.0,419.63,98.55,212.40,0.0,298.8,1029.38,195.58,1224.96
9,10,dt_flex_1_E.DIS_2026,dt_flex,1,E.DIS,4500.0,419.63,76.65,246.15,0.0,298.8,1041.23,197.83,1239.06


## Step 3 — Summary

In [ ]:
# no_flex only (strategies are identical for Base — picked one for readability)
summary = (results[results['strategy'] == 'no_flex']
           [['dso_id', 'cost_energy_net_eur', 'cost_dso_fixed_net_eur',
             'cost_dso_volumetric_net_eur', 'cost_levies_pre_vat_eur',
             'vat_eur', 'total_tcoe_eur']]
           .set_index('dso_id')
           .rename(columns={
               'cost_energy_net_eur'          : 'Spot (net €)',
               'cost_dso_fixed_net_eur'        : 'DSO fix (net €)',
               'cost_dso_volumetric_net_eur'   : 'DSO vol (net €)',
               'cost_levies_pre_vat_eur'       : 'Levies pre-VAT (€)',
               'vat_eur'                       : 'VAT (€)',
               'total_tcoe_eur'                : 'TCoE (€/yr)',
           }))

print(f'TCoE range: {results["total_tcoe_eur"].min()} – {results["total_tcoe_eur"].max()} EUR/year')
print(f'(all strategies identical for Archetype 1)\n')
summary

TCoE range: 1224.96 – 1460.82 EUR/year
(all strategies identical for Archetype 1)



,Spot (net €),DSO fix (net €),DSO vol (net €),Levies pre-VAT (€),VAT (€),TCoE (€/yr)
dso_id,,,,,,
Westnetz,419.63,80.30,428.85,298.8,233.24,1460.82
Bayernwerk,419.63,98.55,212.40,298.8,195.58,1224.96
E.DIS,419.63,76.65,246.15,298.8,197.83,1239.06
Netze BW,419.63,84.00,340.65,298.8,217.18,1360.26
Stromnetz Berlin,419.63,33.36,335.70,298.8,206.62,1294.11
SH Netz,419.63,94.90,288.00,298.8,209.25,1310.58
MITNETZ STROM,419.63,73.00,283.95,298.8,204.32,1279.70


## Step 4 — Export

In [5]:
out = OUTPUTS / 'results_base_2026.csv'
results.to_csv(out, index=False)
print(f'Saved  : {out}')
print(f'Rows   : {len(results)}  (3 strategies × 7 DSOs)')

Saved  : /Users/juliusrucha/Library/CloudStorage/GoogleDrive-rucha.julius@gmail.com/Meine Ablage/Master_Thesis_2026/07_Thesis_GitHub/outputs/results_base_2026.csv
Rows   : 21  (3 strategies × 7 DSOs)
